# Module 1 hook demo, instructor only

Not student-facing. Run this once before Module 1, with a real connection (hotspot is fine), and save the notebook without clearing output. If the room's connection has a bad moment live, the saved output from this morning's run is already on screen to narrate. If the connection holds, re-run the reference agent cell live for the real "watch it happen now" moment.

The naive bot needs no connection at all, it is a plain function. Only the reference agent cell calls a live API.

## Part 1: the naive bot

No API, no internet. Deterministic every time, safe to run live regardless of connectivity.

In [ ]:
def naive_bot(turns):
    """A keyword-matching bot with no memory of its own prior turns.
    Each call only sees the current line, nothing before it."""
    booking = {}
    replies = []
    for turn in turns:
        words = turn.lower().split()
        if "book" in words or "table" in words:
            # crude slot fill from the current line only
            booking = {"party_size": None, "day": None, "time": None}
            for w in words:
                if w.isdigit():
                    booking["party_size"] = w
                if w in ("monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"):
                    booking["day"] = w
                if "pm" in w or "am" in w:
                    booking["time"] = w
            replies.append(f"Booked: {booking}")
        elif "actually" in words or "instead" in words or "make that" in turn.lower():
            # no memory of the earlier booking, so this line alone means nothing to it
            replies.append("Sorry, I didn't understand that. Could you start your booking again?")
        else:
            replies.append("Sorry, I didn't understand that.")
    return replies

conversation = [
    "Book me a table for 4 on Monday at 7pm",
    "actually, make that Tuesday",
]

for turn, reply in zip(conversation, naive_bot(conversation)):
    print(f"user: {turn}")
    print(f"bot:  {reply}\n")

The second line loses the whole booking. There is no state to correct, because there was never any state to begin with, just a reaction to whatever line came in last.

## Part 2: the reference agent

Same two lines. This one holds state across turns and updates only the field that changed. Needs `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` set in `.env`, and a live connection for this cell only.

In [ ]:
import json
import os
from dotenv import load_dotenv

load_dotenv()

SCHEMA = {
    "name": "booking_state",
    "schema": {
        "type": "object",
        "properties": {
            "party_size": {"type": ["integer", "null"]},
            "day": {"type": ["string", "null"]},
            "time": {"type": ["string", "null"]},
        },
        "required": ["party_size", "day", "time"],
    },
}


def reference_agent(turns):
    """Holds one running state object across turns. Each new turn updates only
    the fields the user actually mentioned, the rest carry forward."""
    from openai import OpenAI
    client = OpenAI()
    state = {"party_size": None, "day": None, "time": None}
    replies = []
    for turn in turns:
        prompt = (
            f"Current booking state: {json.dumps(state)}\n"
            f"User just said: \"{turn}\"\n"
            "Return the updated booking state as JSON. Keep any field the user "
            "did not just change."
        )
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_schema", "json_schema": SCHEMA},
        )
        state = json.loads(response.choices[0].message.content)
        replies.append(f"Got it: {state}")
    return replies


for turn, reply in zip(conversation, reference_agent(conversation)):
    print(f"user: {turn}")
    print(f"bot:  {reply}\n")

Same correction, and the day updates while the party size and time carry forward untouched. That difference, state that persists and updates instead of a bot that reacts to one line at a time, is the whole course in miniature.

Next slide: the five-stage arc (`.lu-pipeline`), naming which module builds each stage of what was just shown.